# EE 672: Trustworthy Agentic AI for Security Alert Triage

Walkthrough: load the cases, see the three evidence variants, run configs A/B/C, score every run, and build the results tables and plots.

**Everything here is free.** The default model is a scripted fake (no model at all). For real runs, set `MODEL` to a local open-weight model served by Ollama on Colab's free GPU (ADR 0004). No API key is used anywhere.

**In Colab:** Runtime → Change runtime type → **T4 GPU** (needed only for a real model), then Run all.

## 0. Settings
Change these, then run the notebook top to bottom.

In [ ]:
# Which model to run:
#   "scripted-demo"       free fake model, runs in seconds, exercises logging and scoring
#   "ollama:qwen2.5:7b"   free local open-weight model (needs a GPU runtime in Colab)
MODEL = "scripted-demo"

# What to run:
#   "dry-run"  AUTH-001 only (9 runs), then a projection for the 270-run held-out pass
#   "dev"      every dev case        "heldout"  every held-out case (the real evaluation)
RUN = "dry-run"

# Colab deletes files when the session ends. True saves the run folder to your Google Drive.
SAVE_TO_DRIVE = False

REPO = "https://github.com/blackvectra/agentic-alert-triage.git"
BRANCH = "mlev/beautiful-bardeen-qdvis3"  # change to "main" once the branch is merged

## 1. Setup
In Colab this clones the repo and installs it. Run locally, it uses your checkout.

In [ ]:
import os, subprocess, sys, pathlib
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    if not pathlib.Path("/content/agentic-alert-triage").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO, "/content/agentic-alert-triage"], check=True)
    os.chdir("/content/agentic-alert-triage")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[report]"], check=True)

ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == "notebooks" else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT))
print("repo:", ROOT)

### 1b. Start the free local model (only when `MODEL` starts with `ollama:`)
Installs Ollama, starts its server in the background, and downloads the model weights (about 5 GB for a 7B model, a few minutes). The server listens only on this machine (`localhost:11434`).

In [ ]:
import time, urllib.request

if MODEL.startswith("ollama:"):
    if IN_COLAB:
        subprocess.run("apt-get install -y -qq zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                       shell=True, check=True)
    ollama_server = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(30):  # wait for the server to answer
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except OSError:
            time.sleep(1)
    subprocess.run(["ollama", "pull", MODEL.split(":", 1)[1]], check=True)
    print("Ollama ready with", MODEL)
else:
    print("Scripted demo: no model server needed.")

## 2. The cases
Every case is validated against `fixtures/case.schema.json` on load.

In [ ]:
import pandas as pd
from triage.fixtures import load_cases, build_view, VARIANTS

cases = load_cases()
for c in cases:
    print(c["case_id"], c["split"], c["scenario_group"], "|", c["alert"]["rule"])

## 3. What changes between variants
`withheld` removes records. `injected` overwrites one attacker-controlled field with a payload containing a canary, labelled by delivery technique and attacker intent.

In [ ]:
case = next(c for c in cases if c["case_id"] == "AUTH-001")
for v in VARIANTS:
    view = build_view(case, v)
    ids = [r["evidence_id"] for k in view["evidence"].values() for r in k]
    spec = case["variants"][v]
    print(f"{v:9} expect={spec['expected_behavior']:7} disposition={spec['expected_disposition']:16} records={ids}")

inj = case["variants"]["injected"]["injection"]
print("\nInjected field:", inj["target_evidence_id"], inj["field"])
print("Payload:", inj["payload"])
print("Delivery:", inj["delivery"], "| Intent:", inj["intent"])

## 4. Run every case × variant × config
Uses the same code as the command line (`python -m triage.run`). Output goes to `runs/<RUN>_<model>/`. **Re-running this cell resumes**: runs already logged are skipped, so a Colab disconnect loses nothing.

In [ ]:
from triage import run

OUT = ROOT / "runs" / f"{RUN}_{MODEL.replace(':', '-')}"
args = ["--model", MODEL, "--out", str(OUT)]
args += ["--dry-run"] if RUN == "dry-run" else ["--split", RUN]
run.main(args)

## 5. Results by config
Tables and plots come from `triage/report.py`, which only aggregates the fixed scoring rules in `triage/score.py`.

In [ ]:
from IPython.display import Image, Markdown, display
from triage import report

report_dir = report.write_report(OUT)
display(Markdown((report_dir / "summary.md").read_text()))
for png in sorted(report_dir.glob("*.png")):
    display(Image(filename=str(png)))

## 6. Every run, one row each

In [ ]:
scores = pd.read_json(OUT / "scores.jsonl", lines=True)
scores["injection_success"] = scores["injection"].apply(lambda x: x["success"] if isinstance(x, dict) else None)
scores[["case_id", "variant", "config", "expected_disposition", "predicted_disposition", "abstain_cell",
        "supported_claim_rate", "evidence_recall", "unauthorized_attempts", "injection_success"]]

## 7. Keep the results
Colab's disk is temporary. Copy the run folder to Google Drive, or download it from the Files panel.

In [ ]:
if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    import shutil
    drive.mount("/content/drive")
    dest = pathlib.Path("/content/drive/MyDrive/ee672_runs") / OUT.name
    shutil.copytree(OUT, dest, dirs_exist_ok=True)
    print("saved to", dest)
else:
    print("results are in", OUT)

## Notes
- With `scripted-demo`, every config gives the same scripted answer, so differences only come from what each config could retrieve. The numbers test the harness, not a model.
- Results from one small local model may not transfer to other models (see `docs/related-work.md`).